# 02 — MiniLM Bi-Encoder (Pretrained)
**Roll No:** 23f3004491 | Model 2 of 5 | Milestone 2

A sentence-transformer that maps prompt and options into a shared semantic space, so meaning
matters rather than exact words. Runs on CPU.

In [ ]:
import warnings, re
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"
train = pd.read_csv(f"{BASE}/train.csv")
test  = pd.read_csv(f"{BASE}/test.csv")
OPTIONS = ["A", "B", "C", "D", "E"]

def average_precision_at_3(true_label, predicted_labels):
    for rank, pred in enumerate(predicted_labels[:3]):
        if pred == true_label:
            return 1.0 / (rank + 1)
    return 0.0

def mean_average_precision_at_3(true_labels, predicted_lists):
    return float(np.mean([average_precision_at_3(t, p)
                          for t, p in zip(true_labels, predicted_lists)]))

START_WRAPPERS = ["Pick the best possible answer:", "Select the most accurate option:",
                  "Determine the correct option:", "Identify the correct statement:",
                  "Choose the correct answer:"]

def normalize_core(prompt):
    p = str(prompt).strip()
    for s in START_WRAPPERS:
        if p.startswith(s):
            p = p[len(s):].strip()
    if "?" in p:
        p = p[:p.rfind("?") + 1]
    return re.sub(r"\s+", " ", p).lower().strip()

train["core"] = train["prompt"].apply(normalize_core)
test["core"]  = test["prompt"].apply(normalize_core)

# leakage-free split by unique core question
np.random.seed(42)
cores = train["core"].unique().copy()
np.random.shuffle(cores)
val_cores = set(cores[:200])
valid_df = train[train["core"].isin(val_cores)].drop_duplicates("core").reset_index(drop=True)
print("train:", train.shape, "| validation questions:", len(valid_df))

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

def scores_to_preds(S):
    return [[OPTIONS[j] for j in np.argsort(row)[::-1]] for row in S]

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

def minilm_scores(df):
    p = bi_encoder.encode(df["prompt"].astype(str).tolist(), convert_to_tensor=True,
                          batch_size=64, show_progress_bar=False)
    S = np.zeros((len(df), 5))
    for j, o in enumerate(OPTIONS):
        e = bi_encoder.encode(df[o].astype(str).tolist(), convert_to_tensor=True,
                              batch_size=64, show_progress_bar=False)
        S[:, j] = util.cos_sim(p, e).diagonal().cpu().numpy()
    return S

In [ ]:
preds = scores_to_preds(minilm_scores(valid_df))
score = mean_average_precision_at_3(valid_df["answer"].tolist(), preds)
print(f"MiniLM validation MAP@3: {score:.4f}")

## Observation
MiniLM reaches about 0.40 and rescues 564 questions the TF-IDF pipeline missed entirely,
confirming that semantic similarity beats bag-of-words. Still, it only matches the majority
baseline, because encoding prompt and options separately collapses near-paraphrase options.